![Week 6 Day 2](../assets/banners/w6d2.png)

# Week 6 Day 2 - The Streamlit Dashboard and the Review Queue

Yesterday GradeWise got a front door: a FastAPI backend that uploads, grades in the background, pauses for a human and serves report cards. You drove it with `TestClient`, which is perfect for a developer and useless for Asha.

Today she gets a **dashboard**. We'll build it with [Streamlit](https://streamlit.io/), which turns a Python script into a web app, and we'll spend most of our time on the page that makes GradeWise trustworthy: the **review queue**, where a teacher sees exactly why a mark was flagged and has the final word.

A notebook can't host a web app, so today has three ingredients: the code, explained; screenshots of every page; and something rather neat: we'll run the **real dashboard pages headlessly** from this notebook with Streamlit's own testing tool, against a real API server, and click the review queue's buttons from Python.

![Week 6 roadmap](../assets/roadmap/week6.png)

### 🎯 By the end of this lab you will
1. 🧠 Understand Streamlit's model: a script that re-runs top to bottom on every click, plus `session_state`
2. 🔌 See why the dashboard only talks to GradeWise over HTTP (`gradewise/ui/client.py`)
3. 🗺️ Tour the multipage app: Home, Knowledge base, Exams, Grade a batch, Review queue, Reports, Chat
4. 🧑‍🏫 Walk through the review queue, and use it (headlessly) to override a mark and resume a batch
5. ▶️ Know the commands to run the dashboard yourself

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 2 - Streamlit Admin Dashboard: Knowledge Bases, Exams, Batches | 🧪 |
| 2 | Day 2 - The Review Queue: Approving Flags and Overriding Marks | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">To see the dashboard in your browser, open <b>two terminals</b> in the repo folder:<br/><code>uv run python -m gradewise.api --demo</code> (the API, port 8000)<br/><code>uv run python -m streamlit run gradewise/ui/app.py</code> (the dashboard, <a href="http://localhost:8501">localhost:8501</a>)<br/>Windows PC people: always <code>uv run python -m ...</code>, never <code>uv run streamlit ...</code>: locked-down machines can block freshly created <code>.exe</code> launchers. This notebook needs neither terminal: it starts its own API on a background thread. Everything is offline and free, in 🧪 demo mode.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ No OpenAI API Key set. That's fine this week: every cell runs offline in 🧪 demo mode.")
    print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

## Part 1: Streamlit in five ideas

1. **A page is a script.** `st.title(...)`, `st.button(...)`, `st.dataframe(...)`: each call draws something, in order.
2. **Every interaction re-runs the whole script**, top to bottom. Click a button, and the script runs again with `st.button(...)` returning `True` this once.
3. **`st.session_state` survives re-runs.** It's a dict per browser tab: where you keep "the current batch", "the chat history", "demo mode on".
4. **Forms batch inputs.** Inside `with st.form(...)`, changing a field doesn't re-run anything until the submit button is pressed. The review queue uses one, so typing three marks doesn't trigger three re-runs.
5. **Multipage apps** are a list of `st.Page(...)` objects handed to `st.navigation(...)`, which draws the sidebar menu.

Idea 2 surprises everyone, so let's see it. Here's a three-line counter app. We'll run it with **`AppTest`**, Streamlit's headless test runner: it executes the script exactly as a browser session would, and lets us click buttons from Python.

In [ ]:
from streamlit.testing.v1 import AppTest

COUNTER_APP = """
import streamlit as st

clicks = 0                                                   # 💡 a plain variable: reset on EVERY re-run
st.session_state.setdefault("clicks", 0)                     # 💡 session_state: survives re-runs
if st.button("Mark one more script"):
    clicks += 1
    st.session_state["clicks"] += 1
st.markdown(f"plain variable: {clicks} · session_state: {st.session_state['clicks']}")
"""

at = AppTest.from_string(COUNTER_APP).run()
for _ in range(3):
    at.button[0].click().run()  # 💡 a click = a full re-run of the script
    print(at.markdown[0].value)

The plain variable is born again as `0` on every run, so it never gets past `1`. `session_state` remembers. Every page of the dashboard is written with this in mind: anything that must survive a click (the batch you're following, the chat history, the exam you picked) lives in `st.session_state`.

## Part 2: The dashboard only speaks HTTP

Open `gradewise/ui/client.py`. It's the dashboard's **only** way to reach GradeWise: a thin wrapper around `httpx` with four functions (`get`, `post`, `get_bytes`, `stream_events`) and one error type, `ApiError`, that carries the same what / cause / fix fields as the API's JSON errors.

The UI never imports LangGraph, never opens the database, never loads a model. That's what lets it run in its own container (Day 4), on another machine, or be replaced by a React app without touching the backend.

Let's give the dashboard something to talk to. First a demo API for this notebook (like yesterday), then a **real** uvicorn server for it on a background thread, so the dashboard code talks to it over real HTTP:

In [ ]:
import json
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR, REPO_ROOT
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day2"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:
import logging
import socket
import threading

import uvicorn


def start_api(app, port: int | None = None) -> tuple[str, uvicorn.Server]:
    """Serve `app` with uvicorn on a background thread, on 127.0.0.1 only. Returns (url, server)."""
    if port is None:
        with socket.socket() as s:  # 💡 ask the OS for a free port, so nothing clashes
            s.bind(("127.0.0.1", 0))
            port = s.getsockname()[1]
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning", access_log=False))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.05)
    return f"http://127.0.0.1:{port}", server


for name in ("uvicorn", "uvicorn.access", "uvicorn.error", "httpx", "streamlit"):
    logging.getLogger(name).setLevel(logging.ERROR)  # 💡 keep the notebook quiet

API_URL, server = start_api(app)

from gradewise.ui import client as ui  # noqa: E402  💡 the dashboard's own HTTP client

ui.API_URL = API_URL  # 💡 normally set from the API_URL environment variable (Docker sets it on Day 4)
health = ui.get("/health")
print(f"🔌 Real API serving at {API_URL} · demo={health['demo']} · provider key={health['provider_key']}")

What does the dashboard show when the API is down? Let's point the client at a port where nothing is listening:

In [ ]:
from gradewise.ui.client import ApiError

ui.API_URL = "http://127.0.0.1:9"  # 💡 port 9: nobody home
try:
    ui.get("/health")
except ApiError as err:
    print(f"❌ {err.title}\n   what : {err.payload['what']}")
    print(f"   cause: {err.payload['cause']}\n   fix  : {err.payload['fix']}")
finally:
    ui.API_URL = API_URL

That message is exactly what `app.py` shows instead of a page when the API isn't answering, with the command to start it. A good error tells you what to *do*.

## Part 3: The shell: `gradewise/ui/app.py`

`app.py` runs on every click, before whichever page you're on. In about 80 lines it:

- sets the page config and injects the theme's CSS (`styles.inject()`; colours in `gradewise/ui/.streamlit/config.toml`)
- calls `GET /health` and draws the sidebar: an **API online** pill, the **🧪 Demo mode** toggle, and how many batches are running or waiting for review
- builds the menu: `st.navigation([st.Page("pages/home.py", ...), ...])` with seven pages
- if the API is down, shows the friendly error and stops

The demo toggle is stored as `st.session_state["demo"]`, and **every** API call passes it (`common.demo()`), so one switch flips the whole dashboard between offline fakes and real models. It starts ON when the server has no API key.

<img src="../assets/week6/01_home.png" width="900" alt="🏠 Home: the pipeline as a diagram, lit up by what the API reports, plus 'your first run in four clicks'." style="border-radius:10px;border:1px solid #262F55" />

*🏠 Home: the pipeline as a diagram, lit up by what the API reports, plus 'your first run in four clicks'.*

Let's run the real Home page headlessly and read what it says. It calls three endpoints (`/knowledge-bases`, `/exams`, `/batches`) through `ui.client`, exactly as in the browser:

In [ ]:
import logging

PAGES = REPO_ROOT / "gradewise" / "ui" / "pages"
for logger_name in list(logging.root.manager.loggerDict):  # 💡 keep Streamlit and uvicorn quiet in the notebook
    if logger_name.startswith(("streamlit", "uvicorn")):
        logging.getLogger(logger_name).setLevel(logging.ERROR)


def run_page(name: str, **state) -> AppTest:
    """Run one dashboard page headlessly, with the given session_state (e.g. demo=True)."""
    at = AppTest.from_file(str(PAGES / name), default_timeout=60)
    for key, value in state.items():
        at.session_state[key] = value
    return at.run()


home = run_page("home.py", demo=True)
print("Exceptions:", list(home.exception) or "none")
for md in home.markdown:
    print("•", md.value.splitlines()[0])

## Part 4: Knowledge bases, exams and batches

Each admin page is a **form** that turns into exactly one API call. Rather than duplicate the pages here, we make the same calls they make, through the same client.

### 📚 Knowledge base (`pages/knowledge_base.py`)
A table of collections and chunk counts, then a form: a collection name, a PDF uploader, "also add the CS201 notes", "reset first". Submitting calls:

<img src="../assets/week6/02_knowledge_base.png" width="820" alt="📚 Knowledge base: collections, chunk counts, and the upload-and-ingest form." style="border-radius:10px;border:1px solid #262F55" />

*📚 Knowledge base: collections, chunk counts, and the upload-and-ingest form.*

In [ ]:
kb = ui.post("/knowledge-bases/cs201/documents", data={"include_course_notes": True, "reset": False, "demo": True})
print(f"{kb['name']}: {kb['chunks']} chunks · added {kb['added']} · ≈ ${kb['est_cost_usd']:.4f}")

💡 `ui.post` turns Python booleans into the strings `"true"`/`"false"` that HTML forms send. A small detail, but it's why the same endpoint works from a browser form, from `curl -F`, and from Python.

### 📝 Exams (`pages/exams.py`)
Upload the three PDFs (or toggle "use the sample pack"), press **Extract**, and the page shows the questions with a ✓ when each question's criteria add up to its marks. Nothing is stored until Asha presses **Approve and store**, which resumes the paused exam graph from week 3.

<img src="../assets/week6/03_exams.png" width="820" alt="📝 Exams: the extracted questions with a check that every mark is accounted for, then approve or reject." style="border-radius:10px;border:1px solid #262F55" />

*📝 Exams: the extracted questions with a check that every mark is accounted for, then approve or reject.*

In [ ]:
review = ui.post("/exams", data={"exam_id": "cs_midterm", "use_sample": True, "demo": True})
print(review["status"], "·", review["summary"]["question_count"], "questions")
stored = ui.post("/exams/cs_midterm/approve", json_body={"approved": True, "demo": True})
print(stored["status"], "·", stored["message"])

### ✍️ Grade a batch (`pages/grade.py`)
Pick an exam, choose the answer sheets (synthetic class from transcripts, synthetic handwritten PDFs, or your own uploads), and press **Start grading**. The page then follows the batch **live**: a progress bar per student and an event log. That live stream is tomorrow's topic.

<img src="../assets/week6/04_grade_batch.png" width="820" alt="✍️ Grade a batch: the form, then live progress per student and an event log." style="border-radius:10px;border:1px solid #262F55" />

*✍️ Grade a batch: the form, then live progress per student and an event log.*

In [ ]:
created = ui.post(
    "/batches",
    data={"exam_id": "cs_midterm", "from_transcripts": True, "students": 2, "auto_approve": False, "demo": True},
)
batch_id = created["batch_id"]
batch = wait_for(batch_id)
print(
    f"{batch_id}: {batch['status']} · waiting for {batch['pending']['student_id']} "
    f"({batch['pending']['type']}, {len(batch['pending']['items'])} item(s))"
)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Look at the <b>Auto-approve every review</b> toggle on that page, and its help text: <i>"Skips the human. Fine for a demo, never for real marks."</i> It's off by default, on purpose. The easiest way to undo a human-in-the-loop design is a convenient checkbox that someone ticks once and forgets. Defaults are policy.</span>
        </td>
    </tr>
</table>

### 🎨 A note on charts: `gradewise/ui/styles.py`

The dashboard's tables and bar charts are plain **HTML and inline SVG** (`styles.table()`, `styles.bar_chart()`), not `st.dataframe` or `st.bar_chart`. Two reasons: they match the report cards and slides pixel for pixel, and they need no pandas or pyarrow, so the UI runs even on locked-down Windows machines where those DLLs are blocked. (🧪 TRY: on your own machine, swap one for `st.bar_chart`.) Every value that came from a user or a model goes through `html.escape()` before it reaches `st.html`: model output is untrusted input.

## Part 5: The review queue, where Asha has the last word

This is the page that makes GradeWise responsible. Every batch that paused for a human appears here, and for each flagged answer Asha sees four things side by side:

| What she sees | Where it comes from | Why it matters |
|---|---|---|
| ✍️ **What the student wrote** | the transcription in the checkpointed state | She judges the answer, not the model's summary of it |
| 🏷️ **Why it was flagged** | the guardrails' `review_reason` | "low grader confidence (0.55)" tells her what to look for |
| 🧮 **The grader's reasoning**, criterion by criterion | `CriterionResult.justification` | She can agree or disagree with each step |
| 📎 **The evidence** it cited | `CriterionResult.evidence` | She can check the course notes it relied on |

Then she sets the mark (a `number_input` capped at the question's maximum: principle 7 in the UI too) and presses **Approve and continue**.

<img src="../assets/week6/05_review_queue.png" width="820" alt="🔍 Review queue: the flagged answer, why it was flagged, the grader's reasoning with evidence, and Asha's mark." style="border-radius:10px;border:1px solid #262F55" />

*🔍 Review queue: the flagged answer, why it was flagged, the grader's reasoning with evidence, and Asha's mark.*

Under the hood, the page is short (`gradewise/ui/pages/review.py`, about 140 lines):

1. `GET /batches?status=paused` → the queue (filtered to the current demo/real mode)
2. a selectbox to pick a paused batch, then `styles.stat_cards(...)` for student, review type, flag count, current total
3. inside one `st.form`, for each item: the answer, the reasoning and either a **mark** (`human_review`) or a **transcription text area** (`review_transcription`)
4. only the fields Asha *changed* go into `overrides` / `corrections`
5. `POST /batches/{id}/review` → the API calls `resume_batch(...)` with LangGraph's `Command(resume=...)`, and the page follows the resumed batch live

Let's use it. We run the real page headlessly and look at what Asha would see:

In [ ]:
queue = run_page("review.py", demo=True)
print("Exceptions:", list(queue.exception) or "none")
print("Batch picker :", queue.selectbox[0].value)
for box in queue.number_input:
    print(f"Mark input   : {box.label} = {box.value} (max {box.max})")
print("Button       :", [b.label for b in queue.button])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Asha reads the flagged answer and decides it's worth <b>full marks</b>. We're about to set the first mark input to its maximum and press the button. Predict: what will the API store as that question's <code>review_reason</code>, and what happens to the batch next? (There are two students in it.)</span>
        </td>
    </tr>
</table>

In [ ]:
pending = client.get(f"/batches/{batch_id}").json()["pending"]
student = pending["student_id"]
first = queue.number_input[0]
full_marks = float(first.max)
question_id = first.label.removeprefix("Marks for ")

first.set_value(full_marks)  # 🧑‍🏫 Asha's decision
queue.button[0].click().run()  # ✅ Approve and continue: POST /batches/{id}/review, then follow the stream

print("Exceptions:", list(queue.exception) or "none")
for md in queue.markdown[:3]:
    print("•", md.value)
print("Info:", [i.value for i in queue.info])

The page posted the review, re-ran itself (`st.rerun()`), followed the resumed batch over the event stream, and stopped when the **next** student paused. Let's approve the rest through the API and check what was stored:

In [ ]:
batch = wait_for(batch_id)
while batch["status"] == "paused":
    client.post(f"/batches/{batch_id}/review", json={"approved": True}).raise_for_status()
    batch = wait_for(batch_id)

card = client.get(f"/reports/cs_midterm/{student}").json()
result = next(r for r in card["results"] if r["question_id"] == question_id)
print(f"Batch {batch['status']}. {student} {question_id}: {result['awarded_marks']:g}/{result['max_marks']:g}")
print(f"review_reason: {result['review_reason']}")

The override is stored with its **history** (`teacher override 1.5 → 3`): anyone reading the report card later can see that a human changed this mark, and from what. Every other flagged answer that Asha approved untouched is recorded as "reviewed by teacher". Nothing the model was unsure about reaches a student without a human having looked at it.

### Two kinds of review
- **`human_review`** (marks): `{"overrides": {"Q2b": 3.0}, "approved": true}`
- **`review_transcription`** (handwriting the vision model couldn't read confidently): `{"corrections": {"Q4b": "what the student actually wrote"}, "approved": true}`. In demo mode, any PDF that isn't one of the synthetic students pauses here, so you can try it from the dashboard by uploading any PDF on the Grade page.

<img src="../assets/week6/06_reports.png" width="820" alt="📊 Reports (tomorrow): KPIs, the mark distribution and the hardest questions, then every report card." style="border-radius:10px;border:1px solid #262F55" />

*📊 Reports (tomorrow): KPIs, the mark distribution and the hardest questions, then every report card.*

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">The review queue is where the product's value is decided. Teachers don't adopt a tool that takes control away from them, and schools can't adopt one that can't show its reasoning. A queue that shows <i>only the uncertain answers</i>, with the evidence beside them, is the difference between "marking 30 scripts" and "checking 9 flagged answers": hours saved, with the teacher still accountable. Measure it: track how many flags are approved unchanged vs overridden, and tune the thresholds with data.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The dashboard lives in <code>gradewise/ui/</code>: <code>app.py</code> (the shell and sidebar), <code>client.py</code> (HTTP only), <code>common.py</code> (the demo switch, friendly error cards, the exam picker, live progress), <code>styles.py</code> (the look) and <code>pages/</code> (one file per page). Run it with <code>uv run python -m streamlit run gradewise/ui/app.py</code> (or <code>uv run python -m gradewise serve ui</code>) after starting the API. It finds the API through the <code>API_URL</code> environment variable (default <code>http://localhost:8000</code>).</span>
        </td>
    </tr>
</table>

In [ ]:
server.should_exit = True  # 💡 stop this notebook's API server (Kernel → Restart also does it)
print("🛑 API server stopped")

## Recap, and where we are heading

Today you:
- 🧠 saw Streamlit's re-run model and `session_state` in a three-line app, driven by `AppTest`
- 🔌 learned why the dashboard only speaks HTTP, and saw its friendly "API offline" error
- 🗺️ toured the pages and made the exact API calls their forms make
- 🧑‍🏫 used the real review queue headlessly: overrode a mark, resumed the batch, and found the override in the report card with its history

**Tomorrow (Day 3):** the progress bars that move by themselves. We stream LangGraph's progress to the UI with **Server-Sent Events**, then build the report-card viewer and chat with the results.

⏭ **Next:** `week6/day3.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Add a page to the dashboard. Write a <b>🧗 Question explorer</b> page (a new file, e.g. in <code>output/week6/</code> first) that uses <code>exam_picker()</code>, calls <code>GET /reports/{exam_id}</code>, and draws <code>styles.bar_chart()</code> of each question's class mean % (hardest first) plus a <code>styles.table()</code> of every question with its mean and its <code>flagged</code> count. Test it headlessly with <code>AppTest.from_file(...)</code> against the API from this notebook, then add it to the <code>pages</code> list in <code>app.py</code> on your own machine and see it in the browser. Solution: <code>week6/solutions/day2_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Streamlit concepts (the run model, session state): <a href="https://docs.streamlit.io/develop/concepts">docs.streamlit.io/develop/concepts</a><br/>• Multipage apps with <code>st.navigation</code>: <a href="https://docs.streamlit.io/develop/concepts/multipage-apps">docs.streamlit.io/develop/concepts/multipage-apps</a><br/>• Testing apps with <code>AppTest</code>: <a href="https://docs.streamlit.io/develop/api-reference/app-testing">docs.streamlit.io/develop/api-reference/app-testing</a><br/>• HTTPX: <a href="https://www.python-httpx.org/">python-httpx.org</a></span>
        </td>
    </tr>
</table>